In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import json


def scrape(path):
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(path)



    auction_data = {}

    try:
        auction_name = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((
                By.XPATH, '//div[@class="auction-header row"]/h1'
            ))
        ).text.strip()
        auction_data["auctionname"] = auction_name
    except:
        auction_data["auctionname"] = ""


    try:
        center = driver.find_element(
            By.XPATH, '//li[@class="icon-item icon-item-location"]'
        ).text.strip()
        auction_data["auctioncenter"] = center
    except:
        auction_data["auctioncenter"] = ""

    try:
        time_info = driver.find_element(
            By.XPATH, '//li[@class="icon-item icon-item-time"]'
        ).text.strip()
        auction_data["auctiontime"] = time_info
    except:
        auction_data["auctiontime"] = ""


    print(auction_data)
    if not os.path.exists("database"):
        os.makedirs("database")
    with open("database/db.json", "w") as f:
        json.dump(auction_data, f, indent=4)
    print("✔ Auction header saved")

    # -----------------------------
    # 2. Now extract all car cards
    # -----------------------------
    page = 1
    car_count = 0

    while True:
        print(f"\n📄 Page {page}")

        try:
            # Wait for product cards
            cards = WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located(
                    (By.XPATH, '//div[contains(@class,"product-card-vertical")]')
                )
            )
        except:
            print("No car cards found.")
            break

        for card in cards:

            # LOT NUMBER
            try:
                lot = card.find_element(By.CLASS_NAME, "card-lot").text.strip().replace("Lot ", "")
            except:
                lot = ""

            # REG NUMBER
            try:
                reg = card.find_element(By.CLASS_NAME, "card-reg").text.strip()
            except:
                reg = f"unknown_{car_count}"

            # DETAIL PAGE URL
            try:
                link = card.find_element(By.TAG_NAME, "a").get_attribute("href")
            except:
                continue

            # Open detail page
            driver.execute_script("window.open(arguments[0]);", link)
            driver.switch_to.window(driver.window_handles[1])

            time.sleep(1)

            # Save HTML
            if not os.path.exists("html"):
                os.makedirs("html")

            safe_reg = reg.replace("/", "_").replace("\\", "_")
            filename = f"html/{lot}_{safe_reg}.html"

            with open(filename, "w", encoding="utf-8") as f:
                f.write(driver.page_source)

            print(f"✔ Saved: {filename}")

            # Close detail tab
            driver.close()
            driver.switch_to.window(driver.window_handles[0])

            car_count += 1

        # -----------------------------
        # 3. Go to Next Page
        # -----------------------------
        try:
            next_btn = driver.find_element(
                By.XPATH, '//li[@class="page-item page-item-arrow page-item-arrow-next"]/a'
            )
            next_href = next_btn.get_attribute("href")
            if next_href:
                driver.get(next_href)
                page += 1
                time.sleep(1)
            else:
                break
        except:
            print("No more pages.")
            break

    driver.quit()
    print("\n🎉 Scraping finished!")


# RUN SCRAPER
scrape("https://www.smva.co.uk/auction/27")


{'auctionname': 'Default', 'auctioncenter': 'Swansea Motor Auctions', 'auctiontime': '6pm - 10/12/2025'}
✔ Auction header saved

📄 Page 1
✔ Saved: html/LOT 1_CP65OHJ.html
✔ Saved: html/LOT 2_DG62CFY.html
✔ Saved: html/LOT 3_YR69YTG.html
✔ Saved: html/LOT 60_EY68XDX.html
✔ Saved: html/LOT 61_YE68HKB.html
✔ Saved: html/LOT 64_VO67FJU.html
✔ Saved: html/LOT 65_CU70ULH.html
✔ Saved: html/LOT 66_CN17SXW.html
✔ Saved: html/LOT 67_CP06YTS.html
✔ Saved: html/LOT 68_CE14WTC.html
✔ Saved: html/LOT 69_HV14KHR.html
✔ Saved: html/LOT 70_VK14AHA.html
✔ Saved: html/LOT 71_CA64LCJ.html
✔ Saved: html/LOT 76_YF20SZJ.html
✔ Saved: html/LOT 77_CU15ZTM.html
✔ Saved: html/LOT 78_FT13KSO.html
✔ Saved: html/LOT 79_CA65AVD.html
✔ Saved: html/LOT 81_RV62URN.html
✔ Saved: html/LOT 82_CK67VVW.html
✔ Saved: html/LOT 87_CV66OBM.html
✔ Saved: html/LOT 92_CV72VFD.html
✔ Saved: html/LOT 93_CP15YDX.html
✔ Saved: html/LOT 94_CU21XBD.html
✔ Saved: html/LOT 95_FG68CKF.html
✔ Saved: html/LOT 96_WX15VRP.html
✔ Saved: html/L

In [ ]:
import os,re,json
import csv
import datetime
from bs4 import BeautifulSoup

def fecthDetails(soup):    
    output = {}

    if soup:
        mainDiv = soup.find("ul", class_="details-list")
        if mainDiv:
            items = mainDiv.find_all("li", class_="detail-item")

            for item in items:
                key_tag = item.find("span")
                value_tag = item.find("strong")

                if key_tag and value_tag:
                    key = key_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    output[key] = value

    return output
    
    
def extract_image_urls(soup):
    image_urls = []

    if not soup:
        return ""

    thumbs_div = soup.find("div", class_="ug-thumbs-strip")
    if thumbs_div:
        imgs = thumbs_div.find_all("img", class_="ug-thumb-image")

        for img in imgs:
            src = img.get("src", "")
            if not src:
                continue

       
            src = src.replace(
                "https://dgaww6lqj3.execute-api.eu-west-1.amazonaws.com/prod/buckets/swansea-buckets-s3-public/keys/",
                ""
            )

      
            src = src.replace("/resized/", "/")

    
            if "---" in src:
                src = src.split("---")[0] + ".jpg"

            new_url = f"https://swansea-buckets-s3-public.s3.eu-west-1.amazonaws.com/{src}"

            image_urls.append(new_url)

    return ",".join(image_urls)

def extract_pdf_url(soup):
    if not soup:
        return ""

    pdf_tag = soup.find("a", href=lambda x: x and x.endswith(".pdf"))
    if not pdf_tag:
        return ""

    pdf_path = pdf_tag.get("href")


    base_url = "https://www.smva.co.uk" 

    if pdf_path.startswith("/"):
        return base_url + pdf_path
    else:
        return pdf_path
    
def extract_manual_keys():
    folder = "html"
    output_file = "smva_data.csv"

    keys = [
            "Title",
            "Auction Name",
            "Lot", 
            "Auction type",
            "Center",
            "Make",
            "Model",
            "Variant",
            # "Doors",
            "Reg", 
            "Start Time", 
            "Start Date",
            "D.O.R",
            "Fuel Type",
            "Body Type",
            # "Former Keepers",
            "Transmission",
            "Colour",
            # "MOT Expiry Date",
            "Year",
            "Keys",
            "VAT Status",
            "V5",
            "CAP Clean",
            "CAP Average",
            "CAP Below",
            "CC",
            "Mileage",
            # "Mileage Warranted",
            # "Additional information",
            # "General Condition",
            # "Tyres Condition",
            # "Euro Status",
            # "MOT Due",
            "Inspection Report",
            "Images",
            # "Damaged_images",
            # "Damage_details",
            ]  

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}
            
            title_tag = soup.find("h1",class_="title-h1")
            if title_tag:
                title_text = title_tag.get_text(strip=True)
                row["Title"] = title_text
            else:
                row["Title"] = ""
            lot_tag = soup.find("span" ,class_="pill-item pill-item-lot")
            if lot_tag:
                lot_text = lot_tag.get_text(strip=True).replace("Lot","")
                row["Lot"] = lot_text
            else:
                row["Lot"] = ""
            reg_tag = soup.find("span" ,class_="pill-item pill-item-reg")
            if reg_tag:
                Reg_text = reg_tag.get_text(strip=True)
                row["Reg"] = Reg_text
            else:
                row["Reg"] = ""
            timeandDate_tag = soup.find("span",class_="pill-item pill-item-time")
            timeandDate=timeandDate_tag.get_text(strip=True)
            if timeandDate:
         
                parts = timeandDate.split("-")
                time_part = parts[0].strip()     
                date_part = parts[1].strip()    


                time_24 = datetime.datetime.strptime(time_part, "%I%p").strftime("%H:%M")

                row["Start Time"] = time_24
                row["Start Date"] = date_part
            else:
                row["Start Time"] = ""
                row["Start Date"] = ""

            with open("database/db.json") as datab:
                db = json.load(datab)
            row['Auction Name'] = db.get("auctionname","") 
            row['Center'] = db.get("auctioncenter","") 
            get_details = fecthDetails(soup)
            row["D.O.R"] = get_details.get("Registered")
            row["Body Type"] = get_details.get("Body type")
            row["Make"] = get_details.get("Manufacturer")
            row["Model"] = get_details.get("Model")
            row["Variant"] = get_details.get("Variant")
            row["Fuel Type"] = get_details.get("Fuel")
            row["Transmission"] = get_details.get("Transmission")
            row["V5"] = get_details.get("V5")
            row["VAT Status"] = get_details.get("VAT")
            row["Keys"] = get_details.get("Keys")
            row["Colour"] = get_details.get("Colour")
            row["CAP Clean"] = get_details.get("CAP Clean")
            row["CAP Average"] = get_details.get("CAP Average")
            row["CAP Below"] = get_details.get("CAP Below")
            row[""]
            cc = get_details.get("CC")
            if cc.isdigit():  
                row["CC"] = round(int(cc) / 1000, 1)
            else:
                row["CC"] = "" 
            mil = get_details.get("Mileage", "")
            milage = ""
            if mil:
                milage = "".join(filter(str.isdigit, mil))
            row["Mileage"] = milage
            yearCovert = get_details.get("Registered")
            yearSplite = yearCovert.split("/")
            year = yearSplite[-1] 
            row["Year"] = year
            
            images= extract_image_urls(soup)
            row['Images'] = images if images is not None else ""
            row['Auction type'] = "Online Auction"
            pdfUrl = extract_pdf_url(soup)
            
            row['Inspection Report']=pdfUrl
            
            all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=keys)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()



✔ CSV Generated: smva_data.csv


In [19]:
from urllib.parse import urlparse, urljoin
import threading, requests, os, re
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

df = pd.read_csv("smva_data.csv")

reg_img = df[['Reg', "Images"]]

def add_watermark_to_image(image_path, text="Sourced from Swansea Motor Auctions"):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

        try:
            font = ImageFont.truetype("arial.ttf", 50)
        except:
            font = ImageFont.load_default()

        margin = 10
        bbox = draw.textbbox((0, 0), text, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        x, y = image.width - tw - margin, image.height - th - margin

        draw.rectangle([x - margin, y - margin, x + tw + margin, y + th + margin],
                       fill=(0,0,0,160))

        draw.text((x, y), text, font=font, fill=(255,255,255,200))

        watermarked = Image.alpha_composite(image, txt_layer).convert("RGB")
        watermarked.save(image_path)

        print(f"✔ Watermarked: {image_path}")

    except Exception as e:
        print(f"⚠ Watermark Error: {e}")

def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for index, row in data.iterrows():
        reg_no = str(row["Reg"]).strip()
        if not reg_no:
            continue

        img_urls = [u for u in re.split(r',\s*', str(row["Images"])) if u]
        if not img_urls:
            continue

        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)

        def save_img(url, folder, idx):
            url = url.strip()
            if not url:
                return

            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            parsed = urlparse(url)
            if not parsed.netloc:
                print(f"❌ Invalid URL Skipped: {url}")
                return

            full_path = os.path.join(folder, f"{reg_no}_{idx}.jpg")

            if os.path.exists(full_path):
                print(f"⏩ Skipped (Exists): {full_path}")
                return

            try:
                response = requests.get(url, stream=True, timeout=20)
                response.raise_for_status()

                with open(full_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(full_path)
                print(f"📌 Saved: {full_path}")

            except Exception as e:
                print(f"⚠ Error downloading: {url} -> {e}")

        for i, url in enumerate(img_urls):
            save_img(url, reg_folder, i+1)

def start_funcs():
    t1 = threading.Thread(target=download_images, args=(reg_img,))
    t1.start()
    t1.join()

if __name__ == "__main__":
    start_funcs()


✔ Watermarked: Images\CE12ODW\CE12ODW_1.jpg
📌 Saved: Images\CE12ODW\CE12ODW_1.jpg
✔ Watermarked: Images\CE12ODW\CE12ODW_2.jpg
📌 Saved: Images\CE12ODW\CE12ODW_2.jpg
✔ Watermarked: Images\CE12ODW\CE12ODW_3.jpg
📌 Saved: Images\CE12ODW\CE12ODW_3.jpg
✔ Watermarked: Images\CE12ODW\CE12ODW_4.jpg
📌 Saved: Images\CE12ODW\CE12ODW_4.jpg
✔ Watermarked: Images\CE12ODW\CE12ODW_5.jpg
📌 Saved: Images\CE12ODW\CE12ODW_5.jpg
✔ Watermarked: Images\CE12ODW\CE12ODW_6.jpg
📌 Saved: Images\CE12ODW\CE12ODW_6.jpg
✔ Watermarked: Images\CV12DZZ\CV12DZZ_1.jpg
📌 Saved: Images\CV12DZZ\CV12DZZ_1.jpg
✔ Watermarked: Images\CV12DZZ\CV12DZZ_2.jpg
📌 Saved: Images\CV12DZZ\CV12DZZ_2.jpg
✔ Watermarked: Images\CV12DZZ\CV12DZZ_3.jpg
📌 Saved: Images\CV12DZZ\CV12DZZ_3.jpg
✔ Watermarked: Images\CV12DZZ\CV12DZZ_4.jpg
📌 Saved: Images\CV12DZZ\CV12DZZ_4.jpg
✔ Watermarked: Images\CV12DZZ\CV12DZZ_5.jpg
📌 Saved: Images\CV12DZZ\CV12DZZ_5.jpg
✔ Watermarked: Images\CV12DZZ\CV12DZZ_6.jpg
📌 Saved: Images\CV12DZZ\CV12DZZ_6.jpg
✔ Watermarked: I